In [1]:
import pandas as pd
import polars as pl
from IPython.display import display, clear_output

In [2]:
data = (
    pl.scan_csv('complaints.csv', n_rows=10000)
    .filter(pl.col('Consumer complaint narrative').str.len_chars() > 0)
    .select(["Complaint ID", "Consumer complaint narrative"])
    .collect()
    .to_pandas()
)
display(data)

,Complaint ID,Consumer complaint narrative
0,3642453,These are not my accounts.
1,8113747,Kindly address this issue on my credit report....
2,3573294,"I wrote three requests, the unverified account..."
3,3414709,XXXX XXXX has a old account settled in XXXX th...
4,3584679,They call at all hours and on the weekends usi...
...,...,...
2646,10026411,I have been a victim of identity theft I filed...
2647,2581589,"XXXX XXXX XXXX, XXXX. Pa XXXX I never lived at..."
2648,1379088,Was unable to retrieve my annual credit report...
2649,5678978,I ordered a copy of my report in MyFico and I ...


In [3]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# df = your pandas dataframe

text_col = "Consumer complaint narrative"

# 1. Clean missing values
docs = (
    data[text_col]
    .fillna("")
    .astype(str)
)

# Optional: remove very short complaints
docs = docs[docs.str.split().str.len() >= 5]
docs

0                              These are not my accounts.
1       Kindly address this issue on my credit report....
2       I wrote three requests, the unverified account...
3       XXXX XXXX has a old account settled in XXXX th...
4       They call at all hours and on the weekends usi...
                              ...                        
2646    I have been a victim of identity theft I filed...
2647    XXXX XXXX XXXX, XXXX. Pa XXXX I never lived at...
2648    Was unable to retrieve my annual credit report...
2649    I ordered a copy of my report in MyFico and I ...
2650    I have submitted over 10 dispute letters to ex...
Name: Consumer complaint narrative, Length: 2647, dtype: object

In [4]:
# 2. Convert text to document-term matrix
vectorizer = CountVectorizer(
    lowercase=True,
    stop_words="english",
    max_df=0.95,       # ignore words appearing in >95% of docs
    min_df=5,          # ignore rare words appearing in fewer than 5 docs
    max_features=5000
)

X = vectorizer.fit_transform(docs)

# 3. Fit LDA model
lda = LatentDirichletAllocation(
    n_components=10,   # number of topics
    random_state=42,
    learning_method="batch",
    max_iter=20
)

lda.fit(X)

,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",10
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'batch'
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",20
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",128
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0
,"perp_tol perp_tol: float, default=1e-1Perplexity tolerance. Only used when ``evaluate_every`` is greater than 0.",0.1


In [5]:
feature_names = vectorizer.get_feature_names_out()

def print_topics(model, feature_names, n_words=15):
    for topic_idx, topic in enumerate(model.components_):
        top_word_idx = topic.argsort()[-n_words:][::-1]
        top_words = [feature_names[i] for i in top_word_idx]
        print(f"Topic {topic_idx}: {', '.join(top_words)}")

print_topics(lda, feature_names, n_words=15)

Topic 0: xxxx, company, credit, score, days, loan, car, vehicle, complaint, financial, information, letters, legal, help, breach
Topic 1: credit, information, consumer, identity, report, theft, accounts, reporting, fraudulent, block, agency, section, items, file, inquiries
Topic 2: information, fcra, reporting, account, act, dispute, law, inaccurate, consumer, violation, xxxx, financial, credit, federal, report
Topic 3: credit, late, payment, payments, reporting, report, reported, violation, days, matter, fair, request, error, account, financial
Topic 4: xxxx, xx, account, 00, date, xxxxxxxx, balance, number, opened, credit, inquiry, report, following, request, items
Topic 5: debt, collection, credit, information, provide, account, validation, original, request, reporting, company, proof, creditor, act, violation
Topic 6: consumer, 15, reporting, section, information, 1681, usc, states, account, agency, credit, privacy, rights, written, furnish
Topic 7: account, card, bank, told, xxxx,

In [6]:
topic_probs = lda.transform(X)

df_topics = data.loc[docs.index].copy()
df_topics["lda_topic"] = topic_probs.argmax(axis=1)
df_topics["lda_topic_probability"] = topic_probs.max(axis=1)

df_topics[[text_col, "lda_topic", "lda_topic_probability"]].head()

,Consumer complaint narrative,lda_topic,lda_topic_probability
0,These are not my accounts.,8,0.549961
1,Kindly address this issue on my credit report....,3,0.713820
2,"I wrote three requests, the unverified account...",2,0.698270
3,XXXX XXXX has a old account settled in XXXX th...,8,0.463949
4,They call at all hours and on the weekends usi...,7,0.819985


In [7]:
topic_df = pd.DataFrame(
    topic_probs,
    columns=[f"topic_{i}" for i in range(lda.n_components)],
    index=docs.index
)

df_with_topics = data.loc[docs.index].join(topic_df)
df_with_topics["dominant_topic"] = topic_df.idxmax(axis=1)
display(df_with_topics)

,Complaint ID,Consumer complaint narrative,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9,dominant_topic
0,3642453,These are not my accounts.,0.050000,0.050023,0.050003,0.050005,0.050003,0.050000,0.050005,0.050001,0.549961,0.050000,topic_8
1,8113747,Kindly address this issue on my credit report....,0.005883,0.005885,0.005885,0.713820,0.005883,0.005884,0.005883,0.005883,0.239113,0.005883,topic_3
2,3573294,"I wrote three requests, the unverified account...",0.001064,0.185995,0.698270,0.001064,0.108287,0.001064,0.001064,0.001064,0.001064,0.001064,topic_2
3,3414709,XXXX XXXX has a old account settled in XXXX th...,0.258643,0.003572,0.003572,0.003572,0.252402,0.003573,0.003573,0.003573,0.463949,0.003573,topic_8
4,3584679,They call at all hours and on the weekends usi...,0.020007,0.020000,0.020000,0.020000,0.020000,0.020004,0.020000,0.819985,0.020004,0.020001,topic_7
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2646,10026411,I have been a victim of identity theft I filed...,0.144241,0.226626,0.002703,0.002703,0.249191,0.002703,0.002703,0.002703,0.363723,0.002703,topic_8
2647,2581589,"XXXX XXXX XXXX, XXXX. Pa XXXX I never lived at...",0.178066,0.005557,0.005556,0.005556,0.777480,0.005556,0.005556,0.005559,0.005558,0.005556,topic_4
2648,1379088,Was unable to retrieve my annual credit report...,0.249912,0.003032,0.003031,0.003031,0.003032,0.003031,0.003031,0.380767,0.348101,0.003032,topic_7
2649,5678978,I ordered a copy of my report in MyFico and I ...,0.001852,0.001852,0.001852,0.001852,0.001852,0.424318,0.001852,0.039980,0.522736,0.001852,topic_8
